# W08a Colab B · 확산·조건 생성·최종 앱

생성형 인공지능 · 2026년 10월 19일(월) · 동덕여자대학교 데이터 사이언스 전공 · 유원상 교수

이 노트북은 첫 셀부터 **CPU로 독립 실행**합니다. 다른 노트북이나 지난 실습의 변수가 필요 없습니다. GPU·유료 API·개인 파일 업로드·대형 모델 신규 다운로드가 필수가 아닙니다.

파일 → Drive에 사본 저장 후 사용하세요. 패키지 모듈을 이미 불러왔다면 설치 후 런타임을 재시작합니다. 오류를 숨기거나 가짜 실행 결과로 바꾸지 않습니다.

설명용 작은 수치와 기존 실제 SD 실행 기록을 구별합니다. 기존 기록을 읽었다고 지금 새 모델을 실행했다고 말하지 않습니다. 질문의 미완성 코드는 Markdown으로 제시하고, 본인의 풀이 변수는 `None`으로 남깁니다. **전체 실행 통과는 문제를 모두 풀었다는 뜻이 아닙니다.**

[학생 요약과 문제](https://github.com/lunalab-ai/genAI/tree/2026-fall-w08a/course/notion/w08a) · [전체 공통 API](https://github.com/lunalab-ai/genAI/blob/2026-fall-w08a/src/W08A-API.md)

## 함수 사용 전: 입력·출력·부작용

목록의 한 원소는 픽셀 또는 특징 좌표입니다. 같은 길이끼리 연산하고 `zip(a,b)`는 같은 위치의 값을 짝지습니다. `sum`은 합, `len`은 개수이며, `math.sqrt`는 제곱근, `math.exp`는 자연지수, `math.log`는 자연로그입니다. 행렬의 한 행이 후보 한 개라는 약속을 유지합니다.

표준 문법: [목록·반복](https://docs.python.org/3.13/tutorial/datastructures.html), [math 함수](https://docs.python.org/3.13/library/math.html). 아래 수업 함수는 설치한 고정 태그의 실제 정의 줄에 연결됩니다.


### [schedule_coefficients](https://github.com/lunalab-ai/genAI/blob/2026-fall-w08a/src/luna_genai/review2_core.py#L109)

단계별 beta 목록 → t/beta/alpha/alpha_bar/a/b 표(0-based t).

각 beta는0보다 크고1보다 작다. alpha=1-beta, alpha_bar는 누적곱.
a/b는 신호·잡음 표준편차 계수. 예: [.1,.2] → 누적[.9,.72].
학습이나 샘플러 실행 없이 계수만 계산한다.

### [diffusion_state](https://github.com/lunalab-ai/genAI/blob/2026-fall-w08a/src/luna_genai/review2_core.py#L128)

원본·고정 Gaussian 잡음 실현값·누적alpha → noisy와 세 학습 target.

0<alpha_bar<1. a=sqrt(alpha_bar), b=sqrt(1-alpha_bar), noisy=a*x0+b*eps.
v=a*eps-b*x0는 기존 W06A의 부호 관례. 학습 정답을 만드는 계산이며
학습된 모델 예측/이미지 생성이 아니다. 예: [1],[0],.25 → noisy=[.5].

### [recover_clean](https://github.com/lunalab-ai/genAI/blob/2026-fall-w08a/src/luna_genai/review2_core.py#L143)

같은 길이 noisy·prediction → 원본 추정 목록, 클리핑 없음.

prediction_type=epsilon(기본)/sample/v_prediction, 0<alpha_bar<1.
정답 target이면 대수적으로 원본이 복원된다. 실제 예측 오차는 사라지지 않는다.
예: [.5],[0],.25,'epsilon' → [1.0]. 모델·난수·학습 부작용 없음.

### [cfg_vector](https://github.com/lunalab-ai/genAI/blob/2026-fall-w08a/src/luna_genai/review2_core.py#L164)

두 조건의 같은 상태·시점 잡음 예측과 scale → CFG 결합 예측.

base+s*(conditional-base), 유한 scale>=0, 기본1. 두 배열 길이는 같다.
수학식의0은base/1은conditional. Diffusers API의 <=1 분기와 구별한다.
예: [0,1],[1,0],2 → [2,-1]. 이미지·품질 점수나 새 학습 결과가 아니다.

### [ddim_step](https://github.com/lunalab-ai/genAI/blob/2026-fall-w08a/src/luna_genai/review2_core.py#L177)

교육용 결정론적 DDIM 한 단계 → predicted_x0와previous 배열.

0<alpha_bar<=alpha_previous<=1, eta=0, 클리핑 없음. 누적alpha를 사용.
원본 추정 후 더 낮은 잡음 시점으로 갱신하며 단순 epsilon 빼기가 아니다.
예: [.5],[0],.25,1 → predicted_x0=[1], previous=[1]. 품질 측정 없음.

### [load_review_records](https://github.com/lunalab-ai/genAI/blob/2026-fall-w08a/src/luna_genai/review2_core.py#L194)

기존 W07A SD1.5 기록·25개 파일의 SHA-256 확인 → 기록 dict.

기본은 패키지 assets/w07a. directory는 같은 형식의 별도 검증용 경로.
반환에 current_mode/asset_directory를 추가하되 원파일은 수정하지 않는다.
모든 manifest 파일의 경로/해시를 검사하고 손상·누락 시 명확히 실패한다.
과거 run.mode는 생성 당시 정보다. 현재 실행은 기록 열람이며 모델·GPU·
다운로드·신규 생성 없음. 예: load_review_records()['current_mode'].

### [diffusion_view](https://github.com/lunalab-ai/genAI/blob/2026-fall-w08a/src/luna_genai/review2_app.py#L84)

누적alpha·예측목표 선택 → 좌표표/설명/그림. 0<alpha_bar<1.

clean=[.3,-.7], noise=[-.2,.9]를 사용한다. 세 target은 알려진 원본/잡음에서
만든 정답이다. 정확 target을 역변환한 오차는 모델 성능이 아닌 대수 검산.
epsilon/sample/v_prediction 중 선택, Figure는 호출 후 close.

### [sampling_view](https://github.com/lunalab-ai/genAI/blob/2026-fall-w08a/src/luna_genai/review2_app.py#L110)

CFG 배율 → CFG/독립 DDIM 수치표·설명·그림.

CFG 두 예측은[-.2,.4],[.3,.1]. 별도 DDIM 예는 z=[.4,-.2],
epsilon=[.15,-.25], 누적alpha .36→.81이다. eta=0/클리핑 없음.
CFG 출력은 잡음 예측이며 이미지가 아니다. 수학식0과 실제 API 분기 구별.

### [recorded_view](https://github.com/lunalab-ai/genAI/blob/2026-fall-w08a/src/luna_genai/review2_app.py#L134)

기존 W07A run key → trace표/설정·관측설명/최종이미지 경로.

key=base/negative-red/cfg-4/steps-10. 매 호출 manifest의 모든 해시 검증.
현재는 CPU 기록 읽기이며 신규 생성/재학습/다운로드가 아니다.
seed·초기latent해시·모델revision·설정을 제시하고 norm을 품질로 쓰지 않는다.

### [build_review_app](https://github.com/lunalab-ai/genAI/blob/2026-fall-w08a/src/luna_genai/review2_app.py#L155)

4탭 Gradio 앱 생성, 서버 시작 없음. Q60 summary_fn(record)->str 연결.

record의 mode/changed/observation/limit은 항상 문자열이다. 학생 함수는
이를 문장으로 연결하며 계산·관측값을 만들지 않는다. 예:
app=build_review_app(summary_fn=my_summary); app.launch(share=True).
필수 경로 CPU, 유료API/모델다운로드 없음. 공유 URL은 런타임 수명에 의존.


In [ ]:
import os, sys, subprocess, math
os.environ['GRADIO_ANALYTICS_ENABLED'] = 'False'
IN_COLAB = 'google.colab' in sys.modules
COURSE_REF = '2026-fall-w08a'
if IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps',
        f'luna-genai @ git+https://github.com/lunalab-ai/genAI.git@{COURSE_REF}#subdirectory=src'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
        'gradio==6.26.0', 'matplotlib>=3.8,<4'], check=True)
import luna_genai
assert luna_genai.__version__ == '0.1.13', '패키지 버전이 다릅니다. 설치 후 런타임을 재시작하세요.'
from luna_genai.review_core import definition_link
from luna_genai.review2_core import (reconstruction_stats, vae_quantities, cosine_search,
    precision_at_k, schedule_coefficients, diffusion_state, recover_clean,
    cfg_vector, ddim_step, load_review_records)
from luna_genai.review2_app import (representation_view, search_view, diffusion_view,
    sampling_view, recorded_view, build_review_app)
from IPython.display import display, Markdown, Image
import matplotlib.pyplot as plt
print('Python', sys.version.split()[0], '| course', luna_genai.__version__, COURSE_REF)
print('설명용 수치 계산 및 기존 기록 열람: 새로운 모델 학습/이미지 생성 없음')


### 설치한 구현의 정의 링크 확인

실제 파일의 시작 행을 읽어 같은 고정 태그 링크를 만듭니다. 설치 태그와 API 링크가 일치해야 합니다.


In [ ]:
for fn in [schedule_coefficients, diffusion_state, recover_clean, cfg_vector, ddim_step, load_review_records, diffusion_view, sampling_view, recorded_view, build_review_app]:
    display(Markdown(f"[{fn.__name__}]({definition_link(fn, COURSE_REF)})"))


## 1. 세 과정과 세 예측 목표

전방 계산은 원본과 추가한 잡음으로 noisy 입력을 만듭니다. 학습은 그 입력에서 target을 예측하도록 가중치를 바꿉니다. 샘플링은 고정된 가중치를 사용하여 현재 상태를 바꿉니다.

$x_t=\sqrt{\bar\alpha_t}x_0+\sqrt{1-\bar\alpha_t}\epsilon$입니다. beta와 alpha를 혼동하지 않고, 누적alpha를 직접 계수로 쓰지 않고 제곱근을 취합니다. 다음 target은 학습된 모델 출력이 아니라 알고 있는 원본·잡음으로 구성한 정답입니다. 정확 target 역변환 성공은 대수적 검산입니다.


In [ ]:
display(schedule_coefficients([.04,.09,.16]))
demo_state = diffusion_state([.3,-.7],[-.2,.9],.49)
for demo_kind in ['epsilon','sample','v_prediction']:
    print(demo_kind, demo_state[demo_kind], '원본 복원', recover_clean(demo_state['noisy'],demo_state[demo_kind],.49,demo_kind))
demo_rows, demo_report, demo_figure = diffusion_view(.49,'v_prediction')
display(demo_rows); display(Markdown(demo_report)); display(demo_figure); plt.close(demo_figure)


## 2. CFG와 DDIM 한 단계

같은 latent·시점에서 두 조건의 잡음 예측을 비교합니다. CFG의 $u+s(c-u)$는 잡음 예측을 반환합니다. 배율을 바꾼다고 가중치를 갱신하지 않습니다. 수학식에서 0을 대입하는 것과 실제 Diffusers API의 두갈래 CFG 분기는 구별합니다.

별도 DDIM 예시는 누적alpha .36→.81, eta=0, 클리핑 없음입니다. 원본 추정과 다음 상태를 분리하며 단순히 예측 잡음을 빼지 않습니다.


In [ ]:
demo_rows, demo_report, demo_figure = sampling_view(2.5)
display(demo_rows); display(Markdown(demo_report)); display(demo_figure); plt.close(demo_figure)


## 3. 기존 실제 SD 기록을 읽기

패키지에 포함된 W07A 기록의 파일 해시를 검사합니다. 기존 네 설정은 같은 초기 latent를 사용했습니다. seed 외에 revision·scheduler·steps·조건과 초기 latent 해시를 함께 읽습니다. 사진 하나와 latent norm으로 일반적인 품질 우열을 판단하지 않습니다.

**지금은 기록 열람입니다.** GPU에서 새 이미지를 만들고 싶다면 [기존 W07A GPU 선택 실습](https://colab.research.google.com/github/lunalab-ai/genAI/blob/2026-fall-w07a/notebooks/student/w07a_sampling_app.ipynb)을 별도로 사용합니다. 이 W08a 필수 실행에서는 그 경로를 호출하지 않습니다.


In [ ]:
demo_records = load_review_records()
print(demo_records['current_mode'])
print('기록 생성시각:',demo_records['created_at'], '| 당시 환경:',demo_records['environment'])
demo_trace, demo_report, demo_picture = recorded_view('base')
display(demo_trace[:3]); display(Markdown(demo_report)); display(Image(filename=demo_picture))


## Python 연습 · 본인의 풀이 기록

Notion·PDF와 같은 ID와 조건입니다. 실행 전에 예상하고, 수정 후 변수의 중간값을 확인하세요. 미완료 기록칸은 자동으로 채워지지 않습니다.


### Q30 · 누적곱을 만드는 한 줄

alpha에는 각 단계에서 남기는 비율이 들어 있다. running은 처음부터 현재 단계까지의 누적곱이다. 빈칸에 연산자 하나를 넣고 cumulative의 최종 내용을 써라. 시작값1.0의 이유도 설명하라.

```python
alpha = [0.8, 0.75, 0.5]
running = 1.0
cumulative = []
for value in alpha:
    running = running ___ value
    cumulative.append(running)
print(cumulative)
```

목표: 누적α의 정의를 반복문 상태로 구현한다.

풀이 방법: 실행 전에 각 변수의 값과 길이를 적고, 수정 후 결과의 의미를 설명합니다. 아래 기록칸과 별도의 코드 셀에 직접 작성하세요.


In [ ]:
q30_my_work = None  # 미완료: 본인의 이유와 코드를 별도 셀에 작성하세요.


### Q31 · 잡음 예측 학습의 잘못된 target

model_prediction은 epsilon 예측기로부터 얻은 잡음 예측이다. clean은 원본, epsilon은 실제로 섞은 잡음이다. 아래 loss가 왜 다른 학습 문제인지 설명하고 한 변수를 고쳐 올바른 평균 손실을 구하라. 모델 입력에 epsilon을 추가하는 수정은 허용되지 않는다.

```python
clean = [0.5, -0.5]
epsilon = [-0.4, 0.6]
model_prediction = [-0.1, 0.2]
loss = sum((model_prediction[i] - clean[i]) ** 2 for i in range(2)) / 2
print(loss)
```

목표: 모델의 예측 의미와 loss target을 일치시킨다.

풀이 방법: 실행 전에 각 변수의 값과 길이를 적고, 수정 후 결과의 의미를 설명합니다. 아래 기록칸과 별도의 코드 셀에 직접 작성하세요.


In [ ]:
q31_my_work = None  # 미완료: 본인의 이유와 코드를 별도 셀에 작성하세요.


### Q32 · 두 원소의 노이즈 혼합

math.sqrt는 제곱근이다. 직접 노이즈 식과 일치하도록 빈칸을 채우고 noisy의 두 값을 소수점 넷째 자리까지 구하라. 계산 중 반올림하지 않고 √0.75≈0.8660254를 마지막에 사용할 수 있다. clean과 epsilon은 문제에서 주어진 고정 배열이다.

```python
import math
clean = [1.0, -1.0]
epsilon = [0.5, 1.0]
alpha_bar = 0.25
a = math.sqrt(alpha_bar)
b = math.sqrt(___)
noisy = [a * x + b * e for x, e in zip(clean, epsilon)]
print(noisy)
```

목표: 수식의 두 제곱근 계수를 배열의 같은 위치에 적용한다.

풀이 방법: 실행 전에 각 변수의 값과 길이를 적고, 수정 후 결과의 의미를 설명합니다. 아래 기록칸과 별도의 코드 셀에 직접 작성하세요.


In [ ]:
q32_my_work = None  # 미완료: 본인의 이유와 코드를 별도 셀에 작성하세요.


### Q33 · v 정의의 부호 오류

기존 수업은 v=a·epsilon−b·x0를 사용하고, 역변환은 restored=a·xt−b·v다. 아래 코드의 v 정의를 고쳐 restored가 x0와 같은지 확인하라. 오류 코드의 v와 restored도 구하고, 이것이 신경망 성능 평가인지 설명하라.

```python
a, b = 0.6, 0.8
x0, epsilon = 0.5, -0.25
xt = a * x0 + b * epsilon
v = a * epsilon + b * x0
restored = a * xt - b * v
print(v, restored)
```

목표: 정의·역변환의 부호 일관성을 검산한다.

풀이 방법: 실행 전에 각 변수의 값과 길이를 적고, 수정 후 결과의 의미를 설명합니다. 아래 기록칸과 별도의 코드 셀에 직접 작성하세요.


In [ ]:
q33_my_work = None  # 미완료: 본인의 이유와 코드를 별도 셀에 작성하세요.


### Q34 · 두 픽셀 변환식의 출력 비교

pixels는0부터255까지의 원본 픽셀값이다. 아래 두 줄의 출력을 추적하라. 첫 리스트는 정수 그대로, 둘째는 정확한 분수 또는 소수점 넷째 자리까지 표시한다. 두 식 중 모델이 요구하는 [-1,1] 범위에 맞는 것을 고르고, 그 이유와 범위 변환이 데이터 분산1도 보장하는지 설명하라. 코드 수정은 요구하지 않는다.

```python
pixels = [0, 64, 128, 255]
raw_formula = [2 * p - 1 for p in pixels]
normalized = [2 * (p / 255) - 1 for p in pixels]
print(raw_formula)
print(normalized)
```

목표: 같은 입력에 대한 두 전처리 식의 출력과 척도 전제를 추적한다.

풀이 방법: 실행 전에 각 변수의 값과 길이를 적고, 수정 후 결과의 의미를 설명합니다. 아래 기록칸과 별도의 코드 셀에 직접 작성하세요.


In [ ]:
q34_my_work = None  # 미완료: 본인의 이유와 코드를 별도 셀에 작성하세요.


### Q42 · shape의 원소 수를 반복문으로

배열의 shape가 (배치,채널,높이,너비) 순서로 주어졌다. 라이브러리 없이 전체 원소 수를 계산하도록 빈칸을 채우고, 각 반복 뒤 count를 적어라. 입력은 양의 정수 4개이다.

```python
shape = (3, 4, 16, 16)
count = 1
for size in shape:
    count = count ____ size
print(count)
```

목표: shape의 원소 수 계산을 기본 반복문으로 구현한다.

풀이 방법: 실행 전에 각 변수의 값과 길이를 적고, 수정 후 결과의 의미를 설명합니다. 아래 기록칸과 별도의 코드 셀에 직접 작성하세요.


In [ ]:
q42_my_work = None  # 미완료: 본인의 이유와 코드를 별도 셀에 작성하세요.


### Q43 · 디코딩 직전 척도 복원 오류

latent는 이미 VAE 대표값에 scale을 곱한 확산용 잠재값이다. 디코더는 곱하기 전 척도의 값을 기대한다. 아래 코드에서 잘못된 연산을 고치고 수정 전후 값을 비교하라. scale은 양수이고, 출력의 소수 끝자리 차이는 허용한다.

```python
latent = [0.6, -0.3]
scale = 0.3
decoder_input = [v * scale for v in latent]
print(decoder_input)
```

목표: 디코더 입력의 척도 오류를 고치고 왕복 검산한다.

풀이 방법: 실행 전에 각 변수의 값과 길이를 적고, 수정 후 결과의 의미를 설명합니다. 아래 기록칸과 별도의 코드 셀에 직접 작성하세요.


In [ ]:
q43_my_work = None  # 미완료: 본인의 이유와 코드를 별도 셀에 작성하세요.


### Q44 · 토큰 가중치에서 특징 벡터로

아래는 한 잠재 이미지 위치가 두 텍스트 토큰의 Value를 섞는 교육용 계산이다. 실행 전에 output을 예측하고, j=0과 j=1에서 더하는 두 항을 적어라. weights의 합이 1이라는 사실이 output의 좌표 합도 1임을 뜻하는지 설명하라. 실제 SD에서 측정한 값은 아니다.

```python
weights = [0.25, 0.75]
values = [[2.0, 0.0], [0.0, 4.0]]
output = []
for j in range(2):
    output.append(sum(weights[i] * values[i][j] for i in range(2)))
print(output)
```

목표: 교차 어텐션 Value 가중합을 코드로 추적하고 의미를 설명한다.

풀이 방법: 실행 전에 각 변수의 값과 길이를 적고, 수정 후 결과의 의미를 설명합니다. 아래 기록칸과 별도의 코드 셀에 직접 작성하세요.


In [ ]:
q44_my_work = None  # 미완료: 본인의 이유와 코드를 별도 셀에 작성하세요.


### Q52 · CFG의 좌표별 차이 빈칸

base와 conditioned는 같은 latent·시점에서 서로 다른 조건으로 구한 잡음 예측이다. CFG를 구현하도록 빈칸을 채우고 두 좌표의 중간 차이를 적어라. 입력 리스트 길이는 같고 scale은 실수이다. 결과는 교육용 잡음 예측이다.

```python
base = [0.2, -0.2]
conditioned = [0.6, 0.1]
scale = 2.0
guided = []
for i in range(len(base)):
    guided.append(base[i] + scale * (____))
print(guided)
```

목표: CFG 수식의 같은좌표 차이를 기본 Python으로 구현한다.

풀이 방법: 실행 전에 각 변수의 값과 길이를 적고, 수정 후 결과의 의미를 설명합니다. 아래 기록칸과 별도의 코드 셀에 직접 작성하세요.


In [ ]:
q52_my_work = None  # 미완료: 본인의 이유와 코드를 별도 셀에 작성하세요.


### Q53 · 원본 추정을 다음 상태로 잘못 읽은 코드

아래 step_results는 반환값 이름을 읽는 연습을 위한 독립적인 교육용 결과 표이다. 실제 SD 실행 기록이나 스케줄러 구현이 아니다. 각 사전에서 prev_sample은 다음 생성 상태, pred_original_sample은 그 단계의 원본 추정이라고 정의한다. 다음 생성 상태만 history에 모으려는 코드의 오류를 수정하고, 수정 전후 history를 비교하라. 두 값의 의미를 구분하는 것이 목표이며 API 이름은 문제에 제공되어 있다.

```python
step_results = [
    {"prev_sample": 0.8, "pred_original_sample": 0.4},
    {"prev_sample": 0.6, "pred_original_sample": 0.3},
]
z = 1.0
history = []
for result in step_results:
    z = result["pred_original_sample"]
    history.append(z)
print(history)
```

목표: 스케줄러의 원본 추정과 다음상태 반환값을 구분한다.

풀이 방법: 실행 전에 각 변수의 값과 길이를 적고, 수정 후 결과의 의미를 설명합니다. 아래 기록칸과 별도의 코드 셀에 직접 작성하세요.


In [ ]:
q53_my_work = None  # 미완료: 본인의 이유와 코드를 별도 셀에 작성하세요.


### Q54 · 난수 생성기에 seed 연결하기

이 문제에서 random.Random(k)는 seed k로 새 난수 생성기를 만들고, rng.random()은 그 생성기에서 [0,1) 값 하나를 뽑아 상태를 진행시킨다고 알려 준다. 아래 함수가 호출할 때마다 전달받은 seed로 독립 생성기를 만들도록 빈칸을 채워라. print의 결과와, 같은 생성기 하나에서 연속으로 두 배열을 뽑는 경우와의 차이를 설명하라. 구체적인 난수 숫자 암기는 필요 없다.

```python
import random

def draw(seed):
    rng = random.Random(____)
    return [rng.random() for _ in range(3)]

first = draw(13)
second = draw(13)
print(first == second)
```

목표: seed 인자를 생성기에 실제 연결하고 상태 소비를 설명한다.

풀이 방법: 실행 전에 각 변수의 값과 길이를 적고, 수정 후 결과의 의미를 설명합니다. 아래 기록칸과 별도의 코드 셀에 직접 작성하세요.


In [ ]:
q54_my_work = None  # 미완료: 본인의 이유와 코드를 별도 셀에 작성하세요.


### Q58 · 실행 모드 표의 출력 읽기

아래 표는 이미 확인한 두 활동의 실행 모드를 기록한 것이다. new_run은 해당 활동에서 SD 모델을 실제로 새로 실행했을 때만 True로 적는 불리언이라고 정의한다. 출력과 반복별 new_runs 값을 예상하라. 이 코드 실행 성공을 “새 SD 이미지 두 장 생성”이라고 보고할 수 있는지도 설명하라.

```python
records = [
    {"mode": "기존 실행 기록 열람", "new_run": False},
    {"mode": "설명용 수치 계산", "new_run": False},
]
new_runs = 0
for record in records:
    if record["new_run"]:
        new_runs += 1
print(new_runs)
```

목표: 불리언 조건문을 추적하고 실행 모드의 의미를 설명한다.

풀이 방법: 실행 전에 각 변수의 값과 길이를 적고, 수정 후 결과의 의미를 설명합니다. 아래 기록칸과 별도의 코드 셀에 직접 작성하세요.


In [ ]:
q58_my_work = None  # 미완료: 본인의 이유와 코드를 별도 셀에 작성하세요.


### Q59 · 앱 입력 순서가 바뀐 콜백

앱의 첫 입력은 바꾼 조건, 둘째 입력은 관찰 내용이다. 아래 describe 함수의 인자 순서도 그 계약을 따른다. 코드가 예외 없이 실행되지만 설명의 의미가 틀린 이유를 찾고 호출 한 줄을 수정하라. 원코드와 수정 코드의 출력도 비교하라. 관찰 문자열이 인과관계를 입증한 것인지는 별개이다.

```python
def describe(changed, observed):
    return f"바꾼 조건: {changed} / 관찰: {observed}"

changed = "CFG 3에서 5로"
observed = "경계가 선명해 보임"
message = describe(observed, changed)
print(message)
```

목표: 콜백 입력 순서와 의미를 점검한다.

풀이 방법: 실행 전에 각 변수의 값과 길이를 적고, 수정 후 결과의 의미를 설명합니다. 아래 기록칸과 별도의 코드 셀에 직접 작성하세요.


In [ ]:
q59_my_work = None  # 미완료: 본인의 이유와 코드를 별도 셀에 작성하세요.


### Q60 · 나의 실험 설명 함수를 앱에 연결하기

앱은 summary_fn(record)를 호출하고 반환된 문자열을 화면에 표시한다. record에는 mode, changed, observation, limit 네 키가 항상 있고 값은 모두 문자열이다. 추가 키가 있어도 무시하며 빈 문자열도 그대로 연결한다. 아래 빈칸을 채우고 출력 전체를 쓰라. 이어 완성한 my_summary를 준비된 build_review_app(summary_fn=my_summary)에 연결한다. 이 앱 함수는 계산 결과를 전달할 뿐이며 설명 함수가 모델 생성이나 품질 판정을 새로 수행하는 것은 아니다. 아래 기록은 문항용 가상 기록이다.

```python
def my_summary(record):
    return (
        f"실행 모드: {record['mode']} / "
        f"바꾼 조건: {record['changed']} / "
        f"관찰: {record[____]} / "
        f"한계: {record['limit']}"
    )

record = {
    "mode": "기존 실행 기록 열람",
    "changed": "steps 12에서 6으로",
    "observation": "손잡이 윤곽이 달라짐",
    "limit": "한 쌍의 결과만으로 일반화하지 않음",
}
message = my_summary(record)
print(message)
```

목표: 기록 사전을 읽는 간단한 함수를 앱 callback으로 연결한다.

풀이 방법: 실행 전에 각 변수의 값과 길이를 적고, 수정 후 결과의 의미를 설명합니다. 아래 기록칸과 별도의 코드 셀에 직접 작성하세요.


In [ ]:
def my_summary(record):
    # Q60: mode/changed/observation/limit을 읽어 자신의 설명으로 연결하세요.
    return '현재 조건과 결과를 표에서 확인하고 해석을 작성하세요.'
q60_my_work = None  # 아직 자신의 풀이를 기록하지 않았습니다.


## 독립 실행 검사

아래 검사는 문제은행과 다른 입력의 계산 관계·데이터 무결성·callback 계약을 확인합니다. 연습문제의 답을 공개하거나 자신의 풀이를 자동 채점하지 않습니다.


In [ ]:
check_state=diffusion_state([.12,-.35],[.8,-.2],.37)
for check_kind in ['epsilon','sample','v_prediction']:
    check_restored=recover_clean(check_state['noisy'],check_state[check_kind],.37,check_kind)
    assert all(math.isclose(a,b,abs_tol=1e-12) for a,b in zip(check_restored,[.12,-.35]))
assert cfg_vector([.1,-.2],[.3,.4],0)==[.1,-.2]
check_step=ddim_step([.21,-.13],[.1,.2],.42,.42)
assert all(math.isclose(a,b,abs_tol=1e-12) for a,b in zip(check_step['previous'],[.21,-.13]))
check_records=load_review_records()
assert len(check_records['runs'])==4 and '현재 신규 추론 아님' in check_records['current_mode']
check_rows,check_message,check_figure=diffusion_view(.37,'sample')
assert check_rows and '정답 target' in check_message and '설명용 수치 계산' in check_message
plt.close(check_figure)
print('W08A-B RUNTIME CHECK PASSED')


## 마지막: Q60의 설명 함수를 4탭 앱에 연결

Q60에서 작성한 `my_summary(record)`를 그대로 연결합니다. 실행 모드·바꾼 조건·관찰·한계 네 문자열은 실제 callback이 제공합니다. 본인의 함수가 관측값을 계산하거나 가짜로 만드는 역할은 아닙니다.

준비된 앱의 입력 → 이벤트 → 공통 계산 → 표/그림 → 내 설명 함수 흐름을 확인합니다. Q60을 수정한 뒤 아래 앱 생성 셀을 다시 실행하면 새 함수가 연결됩니다. 10분 활동에서는 탭을 모두 새로 만들지 않고 이 연결 하나를 완성합니다.


In [ ]:
app = build_review_app(summary_fn=my_summary)
assert len(app.fns)==5
print('4개 탭과 5개 이벤트 연결 완료. 실제 브라우저 조작은 별도로 확인합니다.')
if IN_COLAB:
    app.launch(share=True, debug=False, prevent_thread_lock=True)
else:
    print('Local: app built; browser validation is a separate check.')


**실험 순서:** VAE beta만 변경 → 검색 후보만 변경 → 확산 목표만 변경 → CFG 배율만 변경 → 기존 기록 하나 선택 순서로 화면의 실행 모드를 읽습니다. 관찰은 실제 표·그림에 근거하여 Q60 풀이 기록에 남깁니다. 공유 주소는 런타임이 살아 있는 동안 이용할 수 있습니다.


## 풀이 완료와 실행 성공은 별개

`qXX_my_work=None`은 본인의 풀이 기록이 아직 비어 있다는 표시입니다. 자신의 설명과 코드를 작성해도 자동 채점은 수행하지 않습니다. 기록 열람·계산 검사·문제 풀이 완료를 구별하여 복습하세요.


In [ ]:
print('아직 풀이 기록이 없는 문제:', [name for name,value in list(globals().items()) if name.startswith('q') and name.endswith('_my_work') and value is None])
